In [2]:
!pip -q install -U kaggle lightgbm xgboost statsforecast statsmodels scipy
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_e44d3b881862ebb917606063371d567e"
!kaggle competitions download -c m5-forecasting-accuracy -p /content/m5
!cd /content/m5 && unzip -oq m5-forecasting-accuracy.zip
!ls /content/m5

100% 45.8M/45.8M [00:00<00:00, 133MB/s]

calendar.csv		     sales_train_evaluation.csv  sample_submission.csv
m5-forecasting-accuracy.zip  sales_train_validation.csv  sell_prices.csv


In [3]:
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb, time, platform, sys
from scipy import stats
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
D = "/content/m5/"
STORE = "CA_1"
H = 28
T_ALL = 1941
WINDOWS = {"W1": 1801, "W2": 1829, "W3": 1857, "W4": 1913}
VAL_T = 1885
SEEDS = [0]
TRAIN_DAYS = 730
NW_CV = 6
LGB_P = dict(objective="tweedie", tweedie_variance_power=1.1, learning_rate=0.05,
             n_estimators=400, num_leaves=127, min_child_samples=100,
             subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
XGB_P = dict(tree_method="hist", objective="reg:tweedie", tweedie_variance_power=1.1,
             n_estimators=400, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8)

In [4]:
sales = pd.read_csv(D + "sales_train_evaluation.csv")
sales = sales[sales.store_id == STORE].reset_index(drop=True)
cal = pd.read_csv(D + "calendar.csv").iloc[:T_ALL]
pr = pd.read_csv(D + "sell_prices.csv"); pr = pr[pr.store_id == STORE]
n = len(sales)
U = sales[[f"d_{i}" for i in range(1, T_ALL + 1)]].values.astype(np.float32)
P = pr.pivot(index="item_id", columns="wm_yr_wk", values="sell_price") \
      .reindex(index=sales.item_id.values, columns=cal.wm_yr_wk.values).values.astype(np.float32)
VALID = ~np.isnan(P)
R = U * P
R0 = np.nan_to_num(R)
print("series", n, "days", T_ALL, "rows raw", n * T_ALL, "rows with price", int(VALID.sum()),
      "removed (no price)", int((~VALID).sum()), "zero-revenue share", float((R0[VALID] == 0).mean()))

series 3049 days 1941 rows raw 5918109 rows with price 4788267 removed (no price) 1129842 zero-revenue share 0.5520761478004463


In [5]:
def lag(X, k):
    out = np.full_like(X, np.nan, dtype=np.float32); out[:, k:] = X[:, :-k]; return out

def target_feats(X):
    f = {f"lag{k}": lag(X, k) for k in (28, 35, 42)}
    S = pd.DataFrame(X.T).shift(28)
    for w in (7, 28):
        f[f"rm{w}"] = S.rolling(w, min_periods=1).mean().values.T.astype(np.float32)
        f[f"rs{w}"] = S.rolling(w, min_periods=2).std().values.T.astype(np.float32)
    return f

PC = P / lag(P, 7)
snap = cal[f"snap_{STORE[:2]}"].values
cal_f = dict(wday=cal.wday.values, month=cal.month.values, dom=pd.to_datetime(cal.date).dt.day.values,
             event=cal.event_name_1.notna().astype(int).values, snap=snap)
EVENT = np.tile(cal_f["event"], (n, 1)).astype(bool)
SHOCK = EVENT | (np.abs(np.nan_to_num(PC, nan=1.0) - 1) > 1e-6)
sid = np.repeat(np.arange(n), T_ALL); day = np.tile(np.arange(T_ALL), n)
cat_codes = {c: np.repeat(sales[c].astype("category").cat.codes.values, T_ALL)
             for c in ("item_id", "dept_id", "cat_id")}

def build(X):
    d = {"y": X.ravel(), "price": P.ravel(), "pchg": PC.ravel()}
    d.update({k: np.tile(v, n) for k, v in cal_f.items()})
    d.update(cat_codes)
    d.update({k: v.ravel() for k, v in target_feats(X).items()})
    return pd.DataFrame(d)

df_u, df_r = build(U), build(R0)
BASE = ["price", "pchg", "wday", "month", "dom", "event", "snap", "item_id", "dept_id", "cat_id",
        "lag28", "lag35", "lag42", "rm7", "rm28", "rs7", "rs28"]
CATS = ["item_id", "dept_id", "cat_id"]
valid_f = VALID.ravel()

In [6]:
def scale(T):
    s = np.full(n, np.nan)
    for i in range(n):
        y = R0[i, :T]; nz = np.nonzero(y)[0]
        if len(nz) > 1: s[i] = np.mean(np.diff(y[nz[0]:]) ** 2)
    return s

def weights(T):
    w = R0[:, T - 28:T].sum(1); return w / w.sum()

def wrmsse(Y, F, s, w, mask=None):
    se = (Y - F) ** 2
    if mask is None: num = se.mean(1)
    else:
        c = mask.sum(1); num = np.where(c > 0, (se * mask).sum(1) / np.maximum(c, 1), np.nan)
    r = np.sqrt(num / s); ok = ~np.isnan(r) & (s > 0)
    return (w[ok] * r[ok]).sum() / w[ok].sum(), r

def dm_hln(d, h=H):
    d = np.asarray(d); m = len(d); db = d.mean()
    lrv = d.var() + 2 * sum((1 - k / h) * ((d[k:] - db) * (d[:-k] - db)).sum() / m for k in range(1, h))
    if lrv <= 0: return np.nan, np.nan
    st = db / np.sqrt(lrv / m) * np.sqrt((m + 1 - 2 * h + h * (h - 1) / m) / m)
    return st, 2 * (1 - stats.t.cdf(abs(st), m - 1))

In [7]:
def fill(pred, rows):
    F = np.zeros((n, H)); F[sid[rows], day[rows] - T_cur] = np.clip(pred, 0, None); return F

def run_lgb(df, feats, tr, pr_rows, seed, kind="lgb"):
    m = (lgb.LGBMRegressor(**LGB_P, random_state=seed) if kind == "lgb"
         else xgb.XGBRegressor(**XGB_P, random_state=seed))
    kw = {"categorical_feature": CATS} if kind == "lgb" else {}
    m.fit(df.loc[tr, feats], df.loc[tr, "y"], **kw)
    return m.predict(df.loc[pr_rows, feats])

def ets_arrays(T, nw):
    """ETS forecasts for the last nw past origins (training rows of the hybrid) + the 28 test days."""
    lo = max(0, T - TRAIN_DAYS)
    long = pd.DataFrame({"unique_id": np.repeat(np.arange(n), T - lo), "ds": np.tile(np.arange(lo, T), n),
                         "y": R0[:, lo:T].ravel()})
    sf = StatsForecast(models=[AutoETS(season_length=7)], freq=1, n_jobs=-1)
    fc = sf.forecast(df=long, h=H)
    cv = sf.cross_validation(df=long, h=H, step_size=H, n_windows=nw)
    E = np.full((n, T_ALL), np.nan, dtype=np.float32)
    E[fc.unique_id.values, fc.ds.values] = np.nan_to_num(fc["AutoETS"].values)
    E[cv.unique_id.values, cv.ds.values] = np.nan_to_num(cv["AutoETS"].values)
    return E

def run_window(T, seed):
    global T_cur; T_cur = T
    tr = (day < T) & (day >= T - TRAIN_DAYS) & valid_f
    te = (day >= T) & (day < T + H) & valid_f
    out = {}
    out["Naive"] = np.tile(R0[:, T - 1:T], (1, H))
    out["SeasonalNaive"] = np.tile(R0[:, T - 7:T], (1, 4))
    out["LightGBM (direct)"] = fill(run_lgb(df_r, BASE, tr, te, seed), te)
    u = fill(run_lgb(df_u, BASE, tr, te, seed), te)
    out["LightGBM (units x price)"] = u * np.nan_to_num(P[:, T:T + H])
    out["XGBoost (direct)"] = fill(run_lgb(df_r, BASE, tr, te, seed, "xgb"), te)
    return out

def run_stat_and_hybrid(T, seed):
    global T_cur; T_cur = T
    E = ets_arrays(T, NW_CV)
    out = {"ETS": np.nan_to_num(E[:, T:T + H])}
    df_r["ets_fc"] = E.ravel()
    df_r["ets_res"] = (lag(R0, 28) - lag(np.nan_to_num(E, nan=np.nan), 28)).ravel()
    tr = (day < T) & valid_f & ~np.isnan(df_r["ets_fc"].values)
    te = (day >= T) & (day < T + H) & valid_f
    out["Hybrid (LightGBM+ETS)"] = fill(run_lgb(df_r, BASE + ["ets_fc", "ets_res"], tr, te, seed), te)
    out["LightGBM (same rows as hybrid)"] = fill(run_lgb(df_r, BASE, tr, te, seed), te)
    return out

In [8]:
WINDOWS = {"W4": 1913}
TRAIN_DAYS = 365
NW_CV = 3
LGB_P.update(n_estimators=200, num_leaves=63)
XGB_P.update(n_estimators=200)
XGB_P.update(device="cuda")

In [9]:
import pickle
preds, t0 = {}, time.time()
for w, T in WINDOWS.items():
    for seed in SEEDS:
        preds[(w, seed)] = {**run_window(T, seed), **run_stat_and_hybrid(T, seed)}
        print(w, "seed", seed, "done", round((time.time() - t0) / 60, 1), "min")
    pickle.dump(preds, open("/content/preds.pkl", "wb"))

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [09:49:25] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


W4 seed 0 done 20.5 min


In [15]:
rows, per_series, pooled_d = [], {}, {}
for (w, seed), P_ in preds.items():
    T = WINDOWS[w]; Y = R0[:, T:T + H]; s, wt = scale(T), weights(T)
    sh = SHOCK[:, T:T + H] & VALID[:, T:T + H]; nm = (~SHOCK[:, T:T + H]) & VALID[:, T:T + H]
    for name, F in P_.items():
        a, r = wrmsse(Y, F, s, wt); an, _ = wrmsse(Y, F, s, wt, nm); as_, _ = wrmsse(Y, F, s, wt, sh)
        rows.append(dict(window=w, seed=seed, model=name, WRMSSE=a, normal=an, shock=as_,
                         RMSE=np.sqrt(((Y.sum(0) - F.sum(0)) ** 2).mean()), MAE=np.abs(Y.sum(0) - F.sum(0)).mean()))
        if seed == SEEDS[0]: per_series[(w, name)] = (r, wt)
res = pd.DataFrame(rows); res.to_csv("/content/results_long.csv", index=False)
bywin = res.groupby(["model", "window"]).WRMSSE.mean().unstack()
main = pd.DataFrame({"W4 (test)": bywin["W4"], "mean W1-W4": bywin.mean(1), "sd W1-W4": bywin.std(1),
                     "sd over seeds (W4)": res[res.window == "W4"].groupby("model").WRMSSE.std(),
                     "RMSE W4": res[res.window == "W4"].groupby("model").RMSE.mean(),
                     "MAE W4": res[res.window == "W4"].groupby("model").MAE.mean()})
print("\nTABLE main\n", main.round(3))
print("\nRQ1 direct vs units x price\n", bywin.loc[["LightGBM (direct)", "LightGBM (units x price)"]].round(3))
print("windows won by direct:", int((bywin.loc["LightGBM (direct)"] < bywin.loc["LightGBM (units x price)"]).sum()), "of 4")
stat = bywin.loc[["SeasonalNaive", "ETS"]].mean(1).idxmin()
for m in ["LightGBM (direct)", "XGBoost (direct)"]:
    print(f"RQ2 G {m} vs {stat}: {100 * (bywin.loc[stat].mean() - bywin.loc[m].mean()) / bywin.loc[stat].mean():.1f} %")
sh = res.groupby("model")[["normal", "shock"]].mean()
print("\nRQ3\n", sh.loc[["ETS", "LightGBM (same rows as hybrid)", "Hybrid (LightGBM+ETS)"]].round(3))
for k in ["normal", "shock"]:
    a, b = sh.loc["Hybrid (LightGBM+ETS)", k], sh.loc["LightGBM (same rows as hybrid)", k]
    print(f"Delta_{k} = {100 * (a - b) / b:.1f} %")


TABLE main
                                 W4 (test)  mean W1-W4  sd W1-W4  \
model                                                             
ETS                                 0.815       0.817     0.008   
Hybrid (LightGBM+ETS)               0.812       0.820     0.012   
LightGBM (direct)                   0.808       0.815     0.007   
LightGBM (same rows as hybrid)      0.820       0.829     0.012   
LightGBM (units x price)            0.808       0.816     0.009   
Naive                               1.153       1.207     0.065   
SeasonalNaive                       1.060       1.072     0.022   
XGBoost (direct)                    0.819       0.820     0.002   

                                sd over seeds (W4)   RMSE W4    MAE W4  
model                                                                   
ETS                                            NaN  1409.157  1152.971  
Hybrid (LightGBM+ETS)                          NaN  1430.465  1133.007  
LightGBM (direct)       

In [16]:
def daily_diff(A, B):
    d = []
    for w, T in WINDOWS.items():
        Y = R0[:, T:T + H]; s, wt = scale(T), weights(T); ok = (s > 0) & ~np.isnan(s)
        sa, sb = (Y - preds[(w, SEEDS[0])][A]) ** 2, (Y - preds[(w, SEEDS[0])][B]) ** 2
        d.append(((wt[ok] / s[ok])[:, None] * (sa - sb)[ok]).sum(0))
    return np.concatenate(d)

def boot_ci(A, B, B_=1000, seed=0):
    rng = np.random.default_rng(seed); diffs = []
    ra = np.array([per_series[(w, A)][0] for w in WINDOWS]); rb = np.array([per_series[(w, B)][0] for w in WINDOWS])
    wt = np.array([per_series[(w, A)][1] for w in WINDOWS]); ok = ~np.isnan(ra).any(0) & ~np.isnan(rb).any(0)
    for _ in range(B_):
        i = rng.choice(np.where(ok)[0], ok.sum()); ww = wt[:, i] / wt[:, i].sum(1, keepdims=True)
        diffs.append(((ww * ra[:, i]).sum(1) - (ww * rb[:, i]).sum(1)).mean())
    return np.percentile(diffs, [2.5, 97.5])

from statsmodels.stats.multitest import multipletests
pairs = [("LightGBM (direct)", "LightGBM (units x price)"), ("LightGBM (direct)", stat),
         ("Hybrid (LightGBM+ETS)", "LightGBM (same rows as hybrid)")]
pv = []
for A, B in pairs:
    st, p = dm_hln(daily_diff(A, B)); pv.append(p)
    print(f"{A} vs {B}: DM={st:.2f} p={p:.4f} bootstrap 95% CI of WRMSSE diff {boot_ci(A, B).round(3)}")
print("Holm-adjusted p:", multipletests(np.nan_to_num(pv, nan=1.0), method="holm")[1].round(4))

LightGBM (direct) vs LightGBM (units x price): DM=-0.43 p=0.6651 bootstrap 95% CI of WRMSSE diff [-0.005  0.004]
LightGBM (direct) vs ETS: DM=-0.62 p=0.5390 bootstrap 95% CI of WRMSSE diff [-0.01  0.01]
Hybrid (LightGBM+ETS) vs LightGBM (same rows as hybrid): DM=-5.87 p=0.0000 bootstrap 95% CI of WRMSSE diff [-0.013 -0.005]
Holm-adjusted p: [1. 1. 0.]


In [12]:
import lightgbm, xgboost, statsforecast, torch
print("Python", sys.version.split()[0], "| lightgbm", lightgbm.__version__, "| xgboost", xgboost.__version__,
      "| statsforecast", statsforecast.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)",
      "| total time min:", round((time.time() - t0) / 60, 1), "| seeds:", SEEDS)

Python 3.13.16 | lightgbm 4.7.0 | xgboost 3.4.1 | statsforecast 2.1.1 | pandas 2.2.3 | numpy 2.1.3
GPU: Tesla T4 | total time min: 20.5 | seeds: [0]


In [13]:
def ablate(drop):
    global T_cur; T = WINDOWS["W4"]; T_cur = T
    tr = (day < T) & (day >= T - TRAIN_DAYS) & valid_f; te = (day >= T) & (day < T + H) & valid_f
    F = fill(run_lgb(df_r, [f for f in BASE if f not in drop], tr, te, SEEDS[0]), te)
    Y = R0[:, T:T + H]; s, wt = scale(T), weights(T)
    sh_ = SHOCK[:, T:T + H] & VALID[:, T:T + H]
    return wrmsse(Y, F, s, wt)[0], wrmsse(Y, F, s, wt, sh_)[0]
groups = {"full": [], "no price": ["price", "pchg"], "no calendar/event": ["wday", "month", "dom", "event", "snap"],
          "no lag/rolling": ["lag28", "lag35", "lag42", "rm7", "rm28", "rs7", "rs28"]}
ab = {k: ablate(v) for k, v in groups.items()}
for k, (a, s_) in ab.items(): print(f"{k:20s} all {a - ab['full'][0]:+.3f}   shock {s_ - ab['full'][1]:+.3f}")

full                 all +0.000   shock +0.000
no price             all +0.000   shock +0.002
no calendar/event    all +0.017   shock +0.007
no lag/rolling       all +0.018   shock +0.015


In [14]:
for w, T in {"W1": 1801, "W2": 1829, "W3": 1857}.items():
    preds[(w, 0)] = {**run_window(T, 0), **run_stat_and_hybrid(T, 0)}
    print(w, "done")
pickle.dump(preds, open("/content/preds.pkl", "wb"))
WINDOWS = {"W1": 1801, "W2": 1829, "W3": 1857, "W4": 1913}

W1 done
W2 done
W3 done
